# ORF 445 Homework 3


In [61]:
if (!requireNamespace("rkdb", quietly = TRUE)) {
  stop("Package 'rkdb' is required. Install it from https://github.com/KxSystems/rkdb before running this notebook.")
}

library(rkdb)
options(stringsAsFactors = FALSE)

q_date <- function(x) {
  format(as.Date(x), "%Y.%m.%d")
}

q_timespan <- function(hms) {
  paste0("0D", hms, ".000000000")
}

q_symbol_list <- function(syms) {
  syms <- as.character(syms)
  if (length(syms) == 0) {
    stop("Symbol list is empty.")
  }
  paste0("(", paste0("`", syms, collapse = ";"), ")")
}

ensure_sym_col <- function(df) {
  if (!("sym" %in% names(df))) {
    df$sym <- rownames(df)
    rownames(df) <- NULL
  }
  df$sym <- as.character(df$sym)
  df
}

get_course_insts <- function(db) {
  maybe_insts <- try(execute(db, "insts"), silent = TRUE)
  if (!inherits(maybe_insts, "try-error")) {
    insts <- as.character(unlist(maybe_insts, use.names = FALSE))
    insts <- unique(insts[!is.na(insts) & nzchar(insts)])
    if (length(insts) > 0) {
      return(insts)
    }
  }

  message("Server object `insts` not found; using all instruments from instinfo.")
  all_insts <- execute(db, "exec distinct inst from instinfo")
  unique(as.character(unlist(all_insts, use.names = FALSE)))
}

host <- "hfm.princeton.edu"
port <- 6007L

db <- open_connection(host, port)
cat("Connected to", host, "on port", port, "\n")
execute(db, "tables[]")


Connected to hfm.princeton.edu on port 6007 


[1] "instinfo" "quote"    "trade"

## Part (a): Trade Size, Quote Size, Wide-Spread Fraction

We first choose the **most active symbol** for each instrument in `insts` over the analysis window.
Then we compute:

1. Average trade size.
2. Average quote size (`bsiz + asiz`) sampled at trade times via `aj[]`.
3. Fraction of trades where spread is greater than one tick.


In [62]:
dmin_ab <- as.Date("2024-02-05")
dmax_ab <- as.Date("2024-02-23")

insts <- get_course_insts(db)
cat("Number of instruments in analysis set:", length(insts), "\n")

query_vol_by_sym <- paste0(
  "{[d1;d2] select traded_lots:sum siz by inst:sym2inst sym, sym from trade where date within (d1;d2)}[",
  q_date(dmin_ab), ";", q_date(dmax_ab), "]"
)

vol_by_sym <- execute(db, query_vol_by_sym)
vol_by_sym$inst <- as.character(vol_by_sym$inst)
vol_by_sym$sym <- as.character(vol_by_sym$sym)

vol_by_sym <- vol_by_sym[vol_by_sym$inst %in% insts, , drop = FALSE]
vol_by_sym <- vol_by_sym[order(vol_by_sym$inst, -vol_by_sym$traded_lots, vol_by_sym$sym), ]

if (nrow(vol_by_sym) == 0) {
  stop("No trade volume rows were returned for the selected instruments/date range.")
}

active_symbols <- vol_by_sym[!duplicated(vol_by_sym$inst), c("inst", "sym", "traded_lots")]
rownames(active_symbols) <- NULL

cat("Most active symbol picked for each instrument. Example rows:\n")
head(active_symbols, 10)


Number of instruments in analysis set: 23 
Most active symbol picked for each instrument. Example rows:


,inst,sym,traded_lots
,<chr>,<chr>,<int64>
1,BZ,BZF5,2
2,CL,CLF5,1356
3,EMD,EMDH4,138800
4,ES,ESH4,19273180
5,HO,HOF5,540
6,NG,NGF26,2803
7,NIY,NIYG4,5
8,NKD,NKDH4,113908
9,NQ,NQH4,9231477


In [63]:
syms_q <- q_symbol_list(active_symbols$sym)

query_a1 <- paste0(
  "{[d1;d2;slist]",
  " select avg_trade_size:avg siz, total_lots:sum siz, n_trades:count i by sym",
  " from trade",
  " where date within (d1;d2), sym in slist",
  "}[", q_date(dmin_ab), ";", q_date(dmax_ab), ";", syms_q, "]"
)

a1 <- ensure_sym_col(execute(db, query_a1))

query_a2 <- paste0(
  "{[d1;d2;slist]",
  " t:select date,sym,time from trade where date within (d1;d2), sym in slist;",
  " q:select date,sym,time,bsiz,asiz from quote where date within (d1;d2), sym in slist;",
  " j:aj[`date`sym`time; t; q];",
  " select avg_quote_size:avg (bsiz + asiz) by sym from j where not null bsiz, not null asiz",
  "}[", q_date(dmin_ab), ";", q_date(dmax_ab), ";", syms_q, "]"
)

a2 <- ensure_sym_col(execute(db, query_a2))

query_a3 <- paste0(
  "{[d1;d2;slist]",
  " t:select date,sym,time from trade where date within (d1;d2), sym in slist;",
  " q:select date,sym,time,bid,ask from quote where date within (d1;d2), sym in slist;",
  " j:aj[`date`sym`time; t; q];",
  " j:update inst:sym2inst sym from j;",
  " ticks:exec minpxincr by inst from instinfo;",
  " j:update tick:ticks inst from j;",
  " select frac_wide:avg ((ask - bid) > tick) by sym",
  " from j where not null bid, not null ask, not null tick",
  "}[", q_date(dmin_ab), ";", q_date(dmax_ab), ";", syms_q, "]"
)

a3 <- ensure_sym_col(execute(db, query_a3))

part_a <- merge(active_symbols[, c("inst", "sym")], a1[, c("sym", "avg_trade_size")], by = "sym")
part_a <- merge(part_a, a2[, c("sym", "avg_quote_size")], by = "sym")
part_a <- merge(part_a, a3[, c("sym", "frac_wide")], by = "sym")
part_a$quote_trade_ratio <- part_a$avg_quote_size / part_a$avg_trade_size
part_a <- part_a[order(part_a$quote_trade_ratio), ]
rownames(part_a) <- NULL

part_a


ERROR: Error in execute(db, query_a2): Not connected to kdb+ server.


In [ ]:
plot_a <- part_a[
  is.finite(part_a$quote_trade_ratio) & part_a$quote_trade_ratio > 0 &
    is.finite(part_a$frac_wide) & part_a$frac_wide > 0,
  , drop = FALSE
]

plot(
  plot_a$quote_trade_ratio,
  plot_a$frac_wide,
  log = "xy",
  pch = 19,
  col = "navy",
  xlab = "Average quote size / average trade size",
  ylab = "Fraction of trades when spread > 1 tick",
  main = sprintf("Part (a): %s to %s", dmin_ab, dmax_ab)
)

text(
  plot_a$quote_trade_ratio,
  plot_a$frac_wide,
  labels = plot_a$inst,
  pos = 3,
  cex = 0.75
)


## Part (b): Reversion Parameter (Continuations / Reversals)

For each selected symbol, we:

1. Sort trades in event order (`date`, `sym`, `seq`, `time`).
2. Keep nonzero price changes in `prc`.
3. Count adjacent pairs of direction changes:
   - continuation: same sign as previous change
   - reversal: opposite sign
4. Compute `eta = continuations / reversals`.


In [ ]:
query_b <- paste0(
  "{[d1;d2;slist]",
  " select date,time,seq,sym,prc",
  " from trade",
  " where date within (d1;d2), sym in slist",
  "}[", q_date(dmin_ab), ";", q_date(dmax_ab), ";", syms_q, "]"
)

trades_b <- ensure_sym_col(execute(db, query_b))
trades_b <- trades_b[order(trades_b$sym, trades_b$date, trades_b$seq, trades_b$time), ]

calc_eta <- function(prices) {
  dp <- diff(as.numeric(prices))
  dirs <- sign(dp)
  dirs <- dirs[dirs != 0]

  if (length(dirs) < 2) {
    return(c(continuations = NA_real_, reversals = NA_real_, eta = NA_real_))
  }

  continuations <- sum(dirs[-1] == dirs[-length(dirs)])
  reversals <- sum(dirs[-1] != dirs[-length(dirs)])
  eta <- if (reversals > 0) continuations / reversals else NA_real_

  c(continuations = continuations, reversals = reversals, eta = eta)
}

idx_by_sym <- split(seq_len(nrow(trades_b)), trades_b$sym)
reversion_rows <- lapply(names(idx_by_sym), function(sym) {
  vals <- calc_eta(trades_b$prc[idx_by_sym[[sym]]])
  data.frame(sym = sym, continuations = vals[1], reversals = vals[2], eta = vals[3])
})

reversion <- do.call(rbind, reversion_rows)
reversion <- merge(active_symbols[, c("inst", "sym")], reversion, by = "sym", all.x = TRUE)
rownames(reversion) <- NULL

part_b <- merge(part_a[, c("inst", "sym", "quote_trade_ratio")], reversion[, c("sym", "eta")], by = "sym")
part_b <- part_b[order(part_b$quote_trade_ratio), ]
rownames(part_b) <- NULL

reversion


In [ ]:
plot_b <- part_b[
  is.finite(part_b$quote_trade_ratio) & part_b$quote_trade_ratio > 0 &
    is.finite(part_b$eta) & part_b$eta > 0,
  , drop = FALSE
]

plot(
  plot_b$quote_trade_ratio,
  plot_b$eta,
  log = "xy",
  pch = 19,
  col = "firebrick",
  xlab = "Average quote size / average trade size",
  ylab = "Reversion parameter (C/R)",
  main = sprintf("Part (b): %s to %s", dmin_ab, dmax_ab)
)

text(
  plot_b$quote_trade_ratio,
  plot_b$eta,
  labels = plot_b$inst,
  pos = 3,
  cex = 0.75
)


## Part (c): Tick Volatility And Tick Correlation

The functions below match the assignment specification.

- `tickvol(...)` computes volatility vs lag for one or more symbols.
- `tickcorr(...)` computes lagged correlation for two symbols.

Sampling is done on a one-second grid using q `til` + `aj[]`, with window `07:00:00` to `15:00:00`.


In [ ]:
sample_midpoints_1s <- function(db, sym, day, tmin, tmax) {
  query <- paste0(
    "{[d;s;ts;te]",
    " n:1 + (te - ts) div 0D00:00:01;",
    " grid:([] date:n#d; sym:n#s; time:ts + (til n) * 0D00:00:01);",
    " q:select date,sym,time,mid:0.5 * (bid + ask) from quote where date=d, sym=s, time <= te;",
    " aj[`date`sym`time; grid; q]",
    "}[", q_date(day), ";`", sym, ";", q_timespan(tmin), ";", q_timespan(tmax), "]"
  )

  out <- execute(db, query)
  if (is.null(out) || nrow(out) == 0 || !("mid" %in% names(out))) {
    return(numeric(0))
  }
  as.numeric(out$mid)
}

collect_midpoint_series <- function(db, sym, dmin, dmax, tmin, tmax) {
  days <- seq(as.Date(dmin), as.Date(dmax), by = "day")
  mids <- vector("list", length(days))
  names(mids) <- format(days, "%Y-%m-%d")

  for (i in seq_along(days)) {
    mids[[i]] <- sample_midpoints_1s(db, sym, days[i], tmin, tmax)
  }

  mids
}

tickvol <- function(
  db,
  syms,
  dmin, dmax,
  tmin, tmax,
  dtmax = 60
) {
  syms <- as.character(syms)
  out <- data.frame(lag = 1:dtmax)

  mid_cache <- setNames(vector("list", length(syms)), syms)
  for (sym in syms) {
    mid_cache[[sym]] <- collect_midpoint_series(db, sym, dmin, dmax, tmin, tmax)
  }

  for (sym in syms) {
    mids_by_day <- mid_cache[[sym]]
    vol <- rep(NA_real_, dtmax)

    for (k in 1:dtmax) {
      ss <- 0
      n_obs <- 0

      for (mids in mids_by_day) {
        n <- length(mids)
        if (n <= k) {
          next
        }

        d <- mids[(k + 1):n] - mids[1:(n - k)]
        ok <- is.finite(d)
        if (any(ok)) {
          ss <- ss + sum(d[ok]^2)
          n_obs <- n_obs + sum(ok)
        }
      }

      if (n_obs > 0) {
        vol[k] <- sqrt((ss / n_obs) * 3600 / k)
      }
    }

    out[[sym]] <- vol
  }

  out
}

tickcorr <- function(
  db,
  sym1, sym2,
  dmin, dmax,
  tmin, tmax,
  dtmax = 60
) {
  mids1 <- collect_midpoint_series(db, sym1, dmin, dmax, tmin, tmax)
  mids2 <- collect_midpoint_series(db, sym2, dmin, dmax, tmin, tmax)

  out <- data.frame(lag = 1:dtmax, correlation = NA_real_)

  for (k in 1:dtmax) {
    x_all <- numeric(0)
    y_all <- numeric(0)

    for (i in seq_along(mids1)) {
      x <- mids1[[i]]
      y <- mids2[[i]]
      n <- min(length(x), length(y))
      if (n <= k) {
        next
      }

      x <- x[1:n]
      y <- y[1:n]
      dx <- x[(k + 1):n] - x[1:(n - k)]
      dy <- y[(k + 1):n] - y[1:(n - k)]
      ok <- is.finite(dx) & is.finite(dy)

      if (any(ok)) {
        x_all <- c(x_all, dx[ok])
        y_all <- c(y_all, dy[ok])
      }
    }

    if (length(x_all) > 1 && stats::sd(x_all) > 0 && stats::sd(y_all) > 0) {
      out$correlation[k] <- stats::cor(x_all, y_all)
    }
  }

  out
}


In [ ]:
treasury_syms <- c("ZTH4", "ZFH4", "ZNH4", "ZBH4")

fig2_vol <- tickvol(
  db, treasury_syms,
  dmin = "2024-02-05", dmax = "2024-02-23",
  tmin = "07:00:00", tmax = "15:00:00",
  dtmax = 60
)

fig2_corr <- tickcorr(
  db, "ZFH4", "ZNH4",
  dmin = "2024-02-05", dmax = "2024-02-23",
  tmin = "07:00:00", tmax = "15:00:00",
  dtmax = 60
)

op <- par(mfrow = c(2, 1), mar = c(4, 4, 2, 1))

matplot(
  fig2_vol$lag,
  as.matrix(fig2_vol[treasury_syms]),
  type = "l",
  lty = 1,
  lwd = 2,
  col = c("royalblue", "forestgreen", "firebrick", "black"),
  xlab = "Time difference (seconds)",
  ylab = "Volatility (native units per sqrt(hour))",
  main = "Treasury Tick Volatility (07:00 to 15:00)"
)
legend("topright", legend = treasury_syms, lty = 1, lwd = 2,
       col = c("royalblue", "forestgreen", "firebrick", "black"), bty = "n")

plot(
  fig2_corr$lag,
  fig2_corr$correlation,
  type = "l",
  lwd = 2,
  col = "purple4",
  xlab = "Time difference (seconds)",
  ylab = "Correlation",
  main = "Treasury Tick Correlation: ZFH4 vs ZNH4"
)
abline(h = 0, lty = 2, col = "gray50")

par(op)


### Part (c) Explanations For Treasury Products

`ZT < ZF < ZN < ZB` in volatility because longer maturity Treasuries have higher duration, so a given yield move creates larger price moves in native contract units.

The correlation curve illustrates the Epps effect: as lag gets very short, measured correlation falls due to microstructure noise and asynchronous updates; as lag increases, economically meaningful co-movement appears.


In [ ]:
fig3_vol <- tickvol(
  db, c("CLH4", "HOH4"),
  dmin = "2024-02-05", dmax = "2024-02-15",
  tmin = "07:00:00", tmax = "15:00:00",
  dtmax = 60
)

fig3_corr <- tickcorr(
  db, "CLH4", "HOH4",
  dmin = "2024-02-05", dmax = "2024-02-15",
  tmin = "07:00:00", tmax = "15:00:00",
  dtmax = 60
)

op <- par(mfrow = c(2, 1), mar = c(4, 4, 2, 1))

matplot(
  fig3_vol$lag,
  as.matrix(fig3_vol[c("CLH4", "HOH4")]),
  type = "l",
  lty = 1,
  lwd = 2,
  col = c("darkorange3", "steelblue4"),
  xlab = "Time difference (seconds)",
  ylab = "Volatility (native units per sqrt(hour))",
  main = "Energy Tick Volatility (07:00 to 15:00)"
)
legend("topright", legend = c("CLH4", "HOH4"), lty = 1, lwd = 2,
       col = c("darkorange3", "steelblue4"), bty = "n")

plot(
  fig3_corr$lag,
  fig3_corr$correlation,
  type = "l",
  lwd = 2,
  col = "darkgreen",
  xlab = "Time difference (seconds)",
  ylab = "Correlation",
  main = "Energy Tick Correlation: CLH4 vs HOH4"
)
abline(h = 0, lty = 2, col = "gray50")

par(op)


### Part (c) Explanations For Energy Products

Crude oil and heating oil are strongly correlated because heating oil is a refined product derived from crude, so both respond to shared energy supply-demand shocks.

Compared with large-tick Treasury products, heating oil behaves more like a small-tick product in this sample, so bid-ask bounce contributes less artificial short-lag volatility inflation. That is why its short-horizon volatility curve is flatter.


In [64]:
close_connection(db)


[1] 0